In [ ]:
import itertools
import os

# import cmasher as cm
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy


In [ ]:
folder = r"\\192.168.93.141\data\20260820\PO (-) PECD 2W"

data = pd.DataFrame()
for file in sorted(os.listdir(folder)):
    if file.endswith('.h5'):
        angle=float(file.split('_')[2])
        print(file)
        d = pd.read_hdf(os.path.join(folder, file))
        d['angle'] = angle
        # d = d[d['pz'] > 0]
        # d = d[(d['raw_t'] > 200) & (d['raw_t'] < 240)]
        # d['px'] = d['px'] - 0.0015

        # rotate in x-z plane by angle degrees
        # angle = np.radians(d['angle'].iloc[0] + 5)
        # x_new = d['px'] * np.cos(angle) - d['pz'] * np.sin(angle)
        # z_new = d['px'] * np.sin(angle) + d['pz'] * np.cos(angle)
        # d['px'] = x_new
        # d['pz'] = z_new
        # d['py'] = d['py'] + 0.009
        data = pd.concat([data, d], ignore_index=True)
        # d['px'] = -d['px']
        # # d['py']=-d['py']
        # d['pz'] = -d['pz']
        # data = pd.concat([data, d], ignore_index=True)

data['pr'] = np.sqrt(data['px'] ** 2 + data['py'] ** 2 + data['pz'] ** 2)

data = data[data['pr'] < 0.8]


# data = data.sample(frac=0.1)

In [ ]:
for l in range(0, 3):
    for m in range(-l, l + 1):
        data[f'Y{l}{m}'] = scipy.special.sph_harm(m, l, np.arctan2(data['pz'], data['px']),
                                                  np.arccos(data['py'] / data['pr']))
for l in range(0, 3):
    for m in range(-l, l + 1):
        if m == 0:
            data[f'Y{l}{m}_real'] = data[f'Y{l}{m}'].to_numpy().real
        elif m > 0:
            data[f'Y{l}{m}_real'] = np.sqrt(2) * data[f'Y{l}{m}'].to_numpy().real
            data[f'Y{l}{-m}_real'] = np.sqrt(2) * data[f'Y{l}{m}'].to_numpy().imag

means = data.groupby('angle').mean().reset_index()
means_count = data.groupby('angle').count().reset_index()
means_std = data.groupby('angle').std().reset_index()
means_err = means_std / means_count ** 0.5

In [ ]:
sp = data[data['angle'] == 45]
sp = sp[(sp['m/q'] < 580) & (sp['m/q'] < 65)]
sm = data[data['angle'] == -45]
sm = sm[(sm['m/q'] < 580) & (sm['m/q'] < 65)]
spf = len(sp[sp['py'] > 0])
smf = len(sm[sm['py'] > 0])
spb = len(sp[sp['py'] < 0])
smb = len(sm[sm['py'] < 0])
s0= data[(data['angle']==0) & (data['m/q']<60) & (data['m/q']>58)]
s0f = len(s0[s0['py']>0])
s0b = len(s0[s0['py']<0])

PECD = 2 * ((spf - spb) / (spf + smf) - (smf - smb) / (smf + smb))
PECD0 = (s0f - s0b) / (s0f + s0b)
print(PECD)
print(PECD0)

In [ ]:
prs = []
pecds = []
pecd0s = []
pecdms = []
pecdps = []
signals = []
b10p = []
b10m = []
for bin in itertools.pairwise(np.linspace(0, 0.8, 50)):
    sp_bin = sp[(sp['pr'] > bin[0]) & (sp['pr'] <= bin[1])]
    sm_bin = sm[(sm['pr'] > bin[0]) & (sm['pr'] <= bin[1])]
    spf = len(sp_bin[sp_bin['py'] > 0])
    smf = len(sm_bin[sm_bin['py'] > 0])
    spb = len(sp_bin[sp_bin['py'] < 0])
    smb = len(sm_bin[sm_bin['py'] < 0])
    signal = spf + spb + smf + smb

    s0= data[(data['angle']==0) & (data['pr'] > bin[0]) & (data['pr'] <= bin[1])]
    s0f = len(s0[s0['py']>0])
    s0b = len(s0[s0['py']<0])
    if signal > 2000:
        pecds.append(
                2 * ((spf - spb) / (spf + spb) - (smf - smb) / (smf + smb))
        )
        pecd0s.append(
                (s0f - s0b) / (s0f + s0b)
        )
        pecdms.append(
                (smf - smb) / (smf + smb)
        )
        pecdps.append(
                (spf - spb) / (spf + spb)
        )
        prs.append((bin[0] + bin[1]) / 2)
        signals.append(spf + spb + smf + smb)
        b10p.append(sp_bin['Y10_real'].mean())
        b10m.append(sm_bin['Y10_real'].mean())

In [ ]:
#create figure with secondary y axis
fig = go.Figure(
        data=[
            go.Bar(
                    x=prs,
                    y=signals,
                    name='Signal',
                    yaxis='y2',
                    marker_color='gray',
                    opacity=0.5,
            ),
            go.Scatter(
                    x=prs,
                    y=pecds,
                    name='PECD',
                    mode='lines+markers',
                    marker_color='blue',
            ),
            # go.Scatter(
            #         x=prs,
            #         y=pecd0s,
            #         name='PECD (0 deg)',
            #         mode='lines+markers',
            #         marker_color='orange',
            # ),
            # go.Scatter(
            #         x=prs,
            #         y=pecdps,
            #         name='FBA (45 deg)',
            #         mode='lines+markers',
            #         marker_color='red',
            # ),
            # go.Scatter(
            #         x=prs,
            #         y=pecdms,
            #         name='FBA (-45 deg)',
            #         mode='lines+markers',
            #         marker_color='green',
            # ),
        ],
        layout=go.Layout(
                # title='Momentum Resolved PECD for Propylene Oxide',
                xaxis=dict(title='Momentum (a.u.)'),
                yaxis=dict(title='PECD', range=[-0.2, 0.2]),
                yaxis2=dict(
                        title='Signal (counts)',
                        overlaying='y',
                        side='right',
                        showgrid=False,
                ),
                legend=dict(x=0.80, y=0.95),
                height=600,
                width=800,
                template='simple_white+presentation',
        ),
)

fig.update_layout(
        yaxis_showgrid=True,
)
fig.show()
# fig.write_image(
#         "Momentum Resolved PECD Propylene Oxide.png",
#         scale=20
# )

In [ ]:
# create figure with secondary y axis for b10

fig = go.Figure(
        data=[
            go.Bar(
                    x=prs,
                    y=signals,
                    name='Signal',
                    yaxis='y2',
                    marker_color='lightgray',
                    opacity=0.5,
            ),
            go.Scatter(
                    x=prs,
                    y=b10p,
                    name='b10 (Positive Angle)',
                    mode='lines+markers',
                    marker_color='red',
            ),
            go.Scatter(
                    x=prs,
                    y=b10m,
                    name='b10 (Negative Angle)',
                    mode='lines+markers',
                    marker_color='green',
            ),
        ],
        layout=go.Layout(
                title='Momentum Resolved b10 for Propylene Oxide',
                xaxis=dict(title='Momentum (a.u.)'),
                yaxis=dict(title='b10 (a.u.)'),
                yaxis2=dict(
                        title='Signal (counts)',
                        overlaying='y',
                        side='right',
                        showgrid=False,
                ),
                legend=dict(x=0.75, y=0.95),
                height=600,
                width=800,
        ),
)
fig.show()

In [ ]:
px.histogram(
        sm, x="m/q"
)

In [ ]:
data2 = data.copy().sort_values(by=['angle', 'pr'])

In [ ]:
bins = np.linspace(0, 0.8, 100)
data2['pr_bin'] = pd.cut(data2['pr'], bins=bins)

forward_df = data2[data2['py'] > 0]
backward_df = data2[data2['py'] < 0]

hist_df = data2.groupby(['pr_bin', 'angle']).size().reset_index()
hist_df['pr'] = hist_df['pr_bin'].apply(lambda x: x.mid)

px.scatter(
        hist_df,
        x="pr",
        y="angle",
        color=0,
        height=300,
        width=800,
        color_continuous_scale=px.colors.sequential.Inferno,
).update_layout(
        template="plotly_dark",
)

In [ ]:
hist_df_forward = forward_df.groupby(['pr_bin', 'angle']).size()
hist_df_backward = backward_df.groupby(['pr_bin', 'angle']).size()

diff_df = (2 * (hist_df_forward - hist_df_backward) / (hist_df_forward + hist_df_backward)).reset_index()
diff_df['pr'] = hist_df['pr'] = hist_df['pr_bin'].apply(lambda x: x.mid)
diff_df['counts'] = (hist_df_forward + hist_df_backward).reset_index()[0]

px.scatter(
        diff_df,
        x="pr",
        y="angle",
        color=0,
        size="counts",
        height=600,
        width=800,
        color_continuous_scale=px.colors.diverging.RdBu,
        color_continuous_midpoint=0,
        template="plotly_dark",
        range_color=[-0.2, 0.2],
).update_layout(
        coloraxis_colorbar_title="Relative FBA"
)



In [ ]:
px.line(
        diff_df,
        x="pr",
        line_group="angle",
        color=np.abs(diff_df['angle']),
        y=0,
        range_x=[0.1, 0.6],
        range_y=[-0.3, 0.3],
)

In [ ]:
# Fragment Resolved PECD
prs = []
pecds = []
pecd0s = []
pecdms = []
pecdps = []
signals = []
b10p = []
b10m = []
for bin in itertools.pairwise(np.linspace(0, 60, 125)):
    sp_bin = sp[(sp['m/q'] > bin[0]) & (sp['m/q'] <= bin[1])]
    sm_bin = sm[(sm['m/q'] > bin[0]) & (sm['m/q'] <= bin[1])]
    spf = len(sp_bin[sp_bin['py'] > 0])
    smf = len(sm_bin[sm_bin['py'] > 0])
    spb = len(sp_bin[sp_bin['py'] < 0])
    smb = len(sm_bin[sm_bin['py'] < 0])
    signal = spf + spb + smf + smb

    s0= data[(data['angle']==0) & (data['pr'] > bin[0]) & (data['pr'] <= bin[1])]
    s0f = len(s0[s0['py']>0])
    s0b = len(s0[s0['py']<0])
    if signal > 5000:
        pecds.append(
                2 * ((spf - spb) / (spf + spb) - (smf - smb) / (smf + smb))
        )
        # pecd0s.append(
        #         (s0f - s0b) / (s0f + s0b)
        # )
        # pecdms.append(
        #         (smf - smb) / (smf + smb)
        # )
        # pecdps.append(
        #         (spf - spb) / (spf + spb)
        # )
    else:
        pecds.append(np.nan)
    prs.append((bin[0] + bin[1]) / 2)
    signals.append(spf + spb + smf + smb)
    b10p.append(sp_bin['Y10_real'].mean())
    b10m.append(sm_bin['Y10_real'].mean())


In [ ]:
fig = go.Figure(
        data=[
            go.Bar(
                    x=prs,
                    y=signals,
                    name='Signal',
                    yaxis='y2',
                    marker_color='gray',
                    opacity=0.5,
            ),
            go.Scatter(
                    x=prs,
                    y=pecds,
                    name='PECD',
                    mode='lines+markers',
                    marker_color='blue',
            ),
            # go.Scatter(
            #         x=prs,
            #         y=pecd0s,
            #         name='PECD (0 deg)',
            #         mode='lines+markers',
            #         marker_color='orange',
            # ),
            # go.Scatter(
            #         x=prs,
            #         y=pecdps,
            #         name='FBA (45 deg)',
            #         mode='lines+markers',
            #         marker_color='red',
            # ),
            # go.Scatter(
            #         x=prs,
            #         y=pecdms,
            #         name='FBA (-45 deg)',
            #         mode='lines+markers',
            #         marker_color='green',
            # ),
        ],
        layout=go.Layout(
                # title='Fragment Resolved PECD for Propylene Oxide',
                xaxis=dict(title='m/q', range=[0, 60]),
                yaxis=dict(title='PECD', range=[-0.2, 0.2], showgrid=True),
                yaxis2=dict(
                        title='Signal (counts)',
                        overlaying='y',
                        side='right',
                        showgrid=False,
                ),
                legend=dict(x=0.04, y=0.95),
                height=600,
                width=800,
                template="simple_white+presentation"
        ),
)
fig.show()
# fig.write_image(
#         "Fragment Resolved PECD Propylene Oxide.png",
#         scale=20
# )

In [ ]:
fragment=[20,35]
sp_frag=sp[sp['m/q'].between(fragment[0], fragment[1])]
sm_frag=sm[sm['m/q'].between(fragment[0], fragment[1])]


n=30
p_min=0.2
p_max=0.6
sp_frag['pr_bin']=np.digitize(sp_frag['pr'],bins=np.linspace(p_min,p_max,n)).astype(int)
# print(len(sp_frag.groupby('pr_bin').count().reset_index()['px'].to_numpy()))
sm_frag['pr_bin']=np.digitize(sm_frag['pr'],bins=np.linspace(p_min,p_max,n)).astype(int)
# print(len(sm_frag.groupby('pr_bin').count().reset_index()['px'].to_numpy()))

spf=sp_frag[sp_frag['py']>0]
smf=sm_frag[sm_frag['py']>0]
spb=sp_frag[sp_frag['py']<0]
smb=sm_frag[sm_frag['py']<0]


fp=spf.groupby('pr_bin').count()#.reset_index()#['px'].to_numpy()
fm=smf.groupby('pr_bin').count()#.reset_index()#['px'].to_numpy()
bp=spb.groupby('pr_bin').count()#.reset_index()#['px'].to_numpy()
bm=smb.groupby('pr_bin').count()#.reset_index()#['px'].to_numpy()

fp=np.asarray([fp.loc[i,'px'] if i in fp.index else 0 for i in range(1,n-1) ])
fm=np.asarray([fm.loc[i,'px'] if i in fm.index else 0 for i in range(1,n-1) ])
bp=np.asarray([bp.loc[i,'px'] if i in bp.index else 0 for i in range(1,n-1) ])
bm=np.asarray([bm.loc[i,'px'] if i in bm.index else 0 for i in range(1,n-1) ])
# print(len(fp), len(bp), len(fm), len(bm))
px.line(
        x=np.linspace(p_min,p_max,n-2),
        y=fp,
).add_scatter(
        x=np.linspace(p_min,p_max,n-2),
        y=bp,
).add_scatter(
        x=np.linspace(p_min,p_max,n-2),
        y=fm,
).add_scatter(
        x=np.linspace(p_min,p_max,n-2),
        y=bm,
).show()


px.line(
        x=np.linspace(p_min,p_max,n-2),
        y=2*((fp-bp)/(fp+bp)-(fm-bm)/(fm+bm)),
).show()


In [ ]:
fig2 = go.Figure(
        data=[
            go.Bar(
                    x=np.linspace(p_min,p_max,n-2),
                    y=(fp+bp+fm+bm)/2,
                    name='Signal',
                    yaxis='y2',
                    marker_color='lightgray',
                    opacity=0.5,
            ),
            go.Scatter(
                    x=np.linspace(p_min,p_max,n-2),
                    y=2*((fp-bp)/(fp+bp)-(fm-bm)/(fm+bm)),
                    name='PECD',
                    mode='lines+markers',
                    marker_color='blue',
            ),
            ],
            layout=go.Layout(
                title='m/q = 28 Momentum Resolved PECD',
                xaxis=dict(title='pr', range=[p_min, p_max]),
                yaxis=dict(title='PECD', range=[-0.3, 0.3], showgrid=True),
                yaxis2=dict(
                        title='Signal (counts)',
                        overlaying='y',
                        side='right',
                        showgrid=False,
                ),
                legend=dict(x=0.04, y=0.95),
                height=600,
                width=800,
                template="simple_white+presentation"
        ),
)
fig2.add_hline(y=0, line_color='black', line_width=3)
fig2.show()

In [ ]:
gates=[(20,35), (40,50), (53,80)]
def gate_mq(row):
    # print(row)
    for i, (low,high) in enumerate(gates):
        if low<row['m/q']<high:
            return i
    return -1

sp_gated=sp.copy()
sp_gated['gate']=sp_gated.apply(gate_mq, axis=1)
sp_gated['p_gate']=np.digitize(sp_gated['pr'], np.linspace(0,.8,128))
sp_gated['p_bin']=np.linspace(0,.8,128)[sp_gated['p_gate']]
# sp_gated.sort_values('gate', inplace=True)
sp_gated['count']=1
sp_hist= (sp_gated.groupby(['p_bin','gate']).count()['count'].reset_index())
names={"0": "m/q = 28", "1": "m/q=44", "2": "Parent"}
sp_hist.sort_values(['gate','p_bin'], inplace=True)
fig3 = px.line(sp_hist[sp_hist['gate'] != -1], x='p_bin', y='count',color='gate', labels={'p_bin': 'pr'})
fig3.for_each_trace(lambda t: t.update(
        name=names[t.name],
        legendgroup=names[t.name],
))
fig3.show()

In [ ]:
h3d,e=np.histogramdd([
    sp['px'],
    sp['py'],
    sp['pz'],
], bins=50)
xx,yy,zz=e
xx=xx[:-1]
yy=yy[:-1]
zz=zz[:-1]
xx,yy,zz=np.meshgrid(xx,yy,zz)
fig=go.Figure(data=[
    go.Volume(x=xx.flatten(),y=yy.flatten(),z=zz.flatten(),value=h3d.flatten())
]).show()